# Milestone 9a - Build the eval set from production traffic

Milestones 9a to 9d are one project, end to end, in four parts. A support team
runs a small help-center assistant: it searches 24 help articles and answers
customer questions from them. The question is whether a **reranker** would make
it better. To answer that honestly you need everything from milestones 7 and 8,
in order:

| Part | Lifecycle stage | What you do |
|---|---|---|
| **9a (this one)** | Collect, label | Trace real traffic, label it, pull it into an eval set, stratify and split it |
| 9b | Frame, design, build and trace | Write the plan, choose the variants, run them traced |
| 9c | Evaluate | Must-pass gate, headroom, groups, regressions and improvements, flaky questions, the reranker check, the ablation, the decision |
| 9d | Ship, monitor | Canary, spot a new failure in production, fix it, grow the eval set |

`docs/END_TO_END.md` maps every step to its notebook, its Daily Drill lesson
and the experiment plan template.

**The app.** It lives in `src/drill/rag_lab/`, short enough to read. Version
**v1**, the one in production, does a keyword search for the 3 best articles and
asks a model to answer from them. Every request is an **MLflow trace**.

**Do you need Ollama?** It's best with it, running `gemma3:12b`. Without it,
every model reply is replayed from a recording, so the notebook still works.

In [ ]:
import os

import ipytest
import mlflow
import pandas as pd
import pytest
from mlflow.entities import AssessmentSource
from mlflow.genai import datasets

from drill import rag_lab as R

ipytest.autoconfig(clean="test_*")
pd.set_option("display.max_colwidth", 80)

lab = R.Lab()
URI = R.use_mlflow("m9-help-center")
print("Live: answers come from Ollama." if lab.live else "Offline: model replies are replayed from the recording.")
print("MLflow store:", URI)

## 1. Collect: trace what the app does in production

An eval set made of real questions beats one made up at a desk: real customers
phrase things in ways you wouldn't think of. So start from production.

Here, 55 customer questions arrive and v1 answers each one. Each answer is
traced: the question, the articles search found, the model's reply, and how
long each step took. In production you wouldn't know yet which answers were
right; that's what labelling is for.

In [ ]:
production = lab.run(R.QUESTIONS, ["v1"], runs=1, tags={"stage": "production"})

for r in production[:4]:
    print(f"{R.BY_ID[r['qid']]['question']}\n  -> {r['answer'][:150]}\n")
print(f"{len(production)} traced requests. Browse them with: mlflow ui --backend-store-uri {URI}")

## 2. Label: grades and expected answers

Support experts now review the traces. They leave two different kinds of label,
and the difference matters:

- **Feedback** is a grade of *this* answer: was v1 right? It's useful for
  measuring v1 today, and for checking an LLM judge. But it stops meaning anything
  once the app changes and gives a different answer.
- **Expectations** say what *any* right answer must contain: the key facts, and
  which article holds them. Those keep working for every future version. An eval
  set is built from expectations.

For unanswerable questions (the help center doesn't cover them) the expectation
is that the app says it doesn't know.

**Your turn.** The experts' rule for "right" comes down to one check: does the
answer contain every expected fact? Write `has_facts(answer, facts)`. Each fact
may list acceptable wordings separated by `" | "`, like `"3 to 5 | 3-5"`: a fact
counts if any one of them appears. Ignore upper and lower case.

**How to start.** Lower-case the answer once. For each fact, split it on `"|"`,
strip and lower-case each wording, and check whether any of them is in the
answer. If one fact has no match, the answer fails.

In [ ]:
def has_facts(answer, facts):
    """True if every fact (with " | " between acceptable wordings) appears in the answer."""
    # TODO: for each fact, split on "|" and check whether any wording is in the lower-cased answer.
    raise NotImplementedError("Milestone 9a: implement has_facts")

In [ ]:
%%ipytest -qq

def test_all_facts_present():
    assert has_facts("Standard shipping takes 3 to 5 business days.", ["3 to 5 | 3-5", "business days"])


def test_a_missing_fact_fails():
    assert not has_facts("Standard shipping is quick.", ["3 to 5 | 3-5"])


def test_any_wording_counts():
    assert has_facts("It takes 3-5 days", ["3 to 5 | 3-5"])


def test_case_does_not_matter():
    assert has_facts("Use the QR code at Canada Post", ["qr code", "canada post"])


def test_no_facts_means_nothing_to_check():
    assert has_facts("anything", [])

Now the experts label every production trace: a grade as **feedback**, and the
facts and right article as **expectations**. Both are logged onto the trace in
MLflow, with the expert as the source.

In [ ]:
EXPERT = AssessmentSource(source_type="HUMAN", source_id="support-expert")


def expert_grade(answer, q):
    if not q["facts"]:                      # unanswerable: the right answer is "I don't know"
        return R.declined(answer), "should say the help center doesn't cover it"
    ok = not R.declined(answer) and has_facts(answer, q["facts"])
    return ok, "has every expected fact" if ok else "missing an expected fact"


for r in production:
    q = R.BY_ID[r["qid"]]
    good, why = expert_grade(r["answer"], q)
    mlflow.log_feedback(trace_id=r["trace_id"], name="correct", value=good, rationale=why, source=EXPERT)
    mlflow.log_expectation(trace_id=r["trace_id"], name="expected_facts", value=q["facts"], source=EXPERT)
    mlflow.log_expectation(trace_id=r["trace_id"], name="expected_doc", value=q["gold"] or "none", source=EXPERT)

graded = [expert_grade(r["answer"], R.BY_ID[r["qid"]])[0] for r in production]
print(f"v1 in production: {sum(graded)} of {len(graded)} answers right, by the experts' grades")

**A caution about which traces get labelled.** Here the experts labelled every
trace. In real life they label a sample, and how you pick it matters: if they
only see traces a user flagged with a thumbs-down, the eval set fills up with one
kind of problem and misses others. Sample on purpose: some from each kind of
question, some at random, plus every past failure.

## 3. Pull the labelled traces into an eval set

`merge_records` turns traces into dataset records: the question becomes the
record's **inputs**, and the trace's **expectations** come along. Watch what
*doesn't* come along.

In [ ]:
def get_or_create(name):
    try:
        return datasets.get_dataset(name=name)
    except Exception:
        return datasets.create_dataset(name=name)


eval_set = get_or_create("support-eval")
traces = mlflow.search_traces(filter_string=f"tags.session = '{lab.session}'", return_type="list")
eval_set.merge_records(traces)

df = eval_set.to_df()
print(len(df), "records")
df[["inputs", "expectations"]].head(4)

**What happened.** Every record has the question and the expected facts and
article. The experts' **grades** didn't come along: a grade only describes v1's
answer, so it has no place in a set meant to score future versions. If your
experts only ever grade, your eval set ends up with questions and no right
answers. Ask them for expectations too.

## 4. Stratify: check the groups before trusting the set

Group the questions by the kind of thing they ask. Then compare each group's
share of the eval set with its share of real traffic, and work out how big a
difference each group can show, using the ±100 ÷ √n rule from milestone 7.

**Your guess.** Which group is too small to say much on its own?

In [ ]:
GROUP = {q["id"]: q["group"] for q in R.QUESTIONS}
QID = {q["question"]: q["id"] for q in R.QUESTIONS}
counts = pd.Series([GROUP[QID[i["question"]]] for i in df["inputs"]]).value_counts()

table = pd.DataFrame({
    "questions": counts,
    "share of eval set": (counts / counts.sum()).round(2),
    "share of traffic": pd.Series(R.TRAFFIC),
    "margin, about ± points": (100 / counts ** 0.5).round(),
})
table

**What happened.** Fraud is 10% of traffic but about 20% of the eval set: it's
rare and costly, so it gets extra questions on purpose. That's **stratifying**.
Headline scores will be weighted by real traffic later, so the extra questions
don't distort them.

Unanswerable questions are the thinnest group, about ±40 points with 6
questions. That's a known gap: the plan is to add more over time, and 9d shows
how the set grows.

The **must-pass set** is chosen now, before any experiment: the questions that
must never get worse, whatever the average does.

In [ ]:
for qid in R.MUST_PASS:
    print(f"{qid:10} {R.BY_ID[qid]['question']}")

## 5. Split into dev, test and holdout

Three parts, three jobs:

- **dev**: look at it as much as you like while iterating
- **test**: check before a release, rarely, never tune on it
- **holdout**: locked away; opened once to confirm a final decision

Deal each group's questions across the three parts, like cards, so every part
gets every group in proportion. A random split could leave the 6 unanswerable
questions out of the holdout entirely.

**Your turn.** Write `deal_splits(ids_by_group)`. It gets a dict from each group
to its question ids, in order. Deal each group's ids with the repeating pattern
dev, dev, dev, test, holdout. Return a dict from id to split.

**How to start.** Loop over the groups. Inside, `enumerate` gives each id its
position in the group, and `position % 5` picks its slot in the pattern.

In [ ]:
def deal_splits(ids_by_group):
    """Return {id: "dev" | "test" | "holdout"}, dealing each group's ids 3-1-1 in order."""
    # TODO: for each group, give its ids the pattern dev, dev, dev, test, holdout, repeating.
    raise NotImplementedError("Milestone 9a: implement deal_splits")

In [ ]:
%%ipytest -qq

def test_deals_each_group_on_its_own():
    split = deal_splits({"billing": ["b1", "b2", "b3", "b4", "b5"], "fraud": ["f1", "f2"]})
    assert split == {"b1": "dev", "b2": "dev", "b3": "dev", "b4": "test", "b5": "holdout",
                     "f1": "dev", "f2": "dev"}


def test_the_pattern_repeats():
    split = deal_splits({"faq": [f"q{i}" for i in range(7)]})
    assert [split[f"q{i}"] for i in range(7)] == ["dev", "dev", "dev", "test", "holdout", "dev", "dev"]


def test_no_groups():
    assert deal_splits({}) == {}

In [ ]:
ids_by_group = {}
for q in R.QUESTIONS:
    ids_by_group.setdefault(q["group"], []).append(q["id"])

SPLIT = deal_splits(ids_by_group)
assert SPLIT == R.default_split()   # 9b-9d use this same split

pd.crosstab(pd.Series(GROUP), pd.Series(SPLIT), margins=True)

Last step: record each question's group, split and must-pass flag on the dataset
itself, as tags. Merging the same questions again updates them rather than
adding duplicates.

In [ ]:
eval_set.merge_records([
    {"inputs": {"question": q["question"]},
     "expectations": {"expected_facts": q["facts"], "expected_doc": q["gold"] or "none"},
     "tags": {"qid": q["id"], "group": q["group"], "split": SPLIT[q["id"]],
              "must_pass": "yes" if q["id"] in R.MUST_PASS else "no"}}
    for q in R.QUESTIONS
])
final = eval_set.to_df()
print(len(final), "records; tags on the first one:", {k: v for k, v in final.iloc[0]["tags"].items() if k != "mlflow.user"})

## What you have now

- **An eval set of 55 real questions**, each with the facts a right answer needs
  and the article that holds them, versioned in MLflow as `support-eval`.
- **Groups**, compared with real traffic, with the thin one noted.
- **A must-pass set** of 6 questions, chosen before any experiment.
- **A dev / test / holdout split**, dealt group by group.
- **v1's baseline**, from the experts' grades.

That's the ground everything else stands on. In **9b** you'll use it to plan
and run the reranker experiment.